In [1]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import time
from pandasql import sqldf
os.chdir(r"D:\Service Systems upd\Service-system-research")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)

In [2]:
merged_df = pd.read_csv(
    "df_1_not_merged_2_merged.csv",
)
usecols=[col for col in merged_df.columns.tolist()]

In [3]:
merged_df['event_type'].value_counts()

event_type
1    1946105
2    1676473
7     266809
Name: count, dtype: int64

In [2]:
#Loading relevant dfs:
df_old = pd.read_csv('merged_session_events.csv')
df_after_stage1 = pd.read_csv('agents_merged_fixed_28_5.csv')
# agg_df = pd.read_csv('aggregated_df_11_5.csv')
# df_exploded = pd.read_csv("df_exploded_all_data.csv")
df_reg_before = pd.read_csv('df_choicesets_29_05_2026.csv')
# df_reg_after = pd.read_csv('choicesets_after_customers_msg_unified_21_03.csv')

In [7]:
df_session = pd.read_csv('merged_session.csv')

In [ ]:
# adding chosen_time column for comfort
# df_reg_before['chosen_time'] = df_reg_before['end_time'] + df_reg_before['waiting_time']
# df_reg_after['chosen_time'] = df_reg_after['end_time'] + df_reg_after['waiting_time']
# df_reg_before['chosen_date'] = pd.to_datetime(df_reg_before['chosen_time'], unit = 's')
# df_reg_after['chosen_date'] = pd.to_datetime(df_reg_after['chosen_time'], unit = 's')

In [ ]:
# size_df = df_reg_before.groupby('choice_set').size().reset_index(name= 'size')

# df_reg_size = df_reg_before.merge(size_df, on = 'choice_set', how = 'left')

# print(df_reg_before.shape[0], df_reg_size.shape[0])

# df_reg_size.to_csv('df_choicesets_06_05_2026.csv', index = False)

2342263 2342263


In [6]:
# Choosing a problematic choice_set
df_reg_before[df_reg_before['chosen_time'] == 1495936117]

,id_site,id_session,id_visitor,id_rep,start_date,start_time,end_date,end_time,duration,number_words,number_chars,number_lines,answer_canned,sentiment,accept_date,accept_time,read_date,read_time,event_type,event_type_desc,outcome,outcome_desc,subsession,delay,id_rep_code,sentiment_type,id_agent,id_agent_code,source_file,event_id,choice_set,chosen,waiting_time,workload,chosen_time,chosen_date
200285,1,100165380,200058241,30000067,28/05/2017 01:48:21,1495936101,28/05/2017 01:48:24,1495936104,0,11,50,0,3,0,01/01/1970,0,01/01/1970,0,1,visitor_line,1,served,1,0,2,3,30000067,68,D28052017,3891655,144290,1,13,11.0,1495936117,2017-05-28 01:48:37
1545678,1,100144015,200077098,30000683,28/05/2017 01:47:41,1495936061,28/05/2017 01:48:21,1495936101,40,7,31,0,2,0,28/05/2017 01:33:37,1495935217,28/05/2017 01:33:37,1495935217,1,visitor_line,1,served,1,0,684,3,30000683,684,D28052017,3539233,1079603,0,16,10.0,1495936117,2017-05-28 01:48:37
1545679,1,100209964,200099525,30000683,28/05/2017 01:47:15,1495936035,28/05/2017 01:47:41,1495936061,26,24,120,0,2,-2,28/05/2017 01:47:18,1495936038,28/05/2017 01:47:26,1495936046,1,visitor_line,1,served,1,0,684,2,30000683,684,D28052017,4403900,1079603,1,56,10.0,1495936117,2017-05-28 01:48:37


In [13]:
df_check = df_reg_before.sort_values(
    by=["choice_set", "end_date"],
    ascending=True
).copy()

df_check["prev_id_session"] = df_check.groupby("choice_set")["id_session"].shift(1)
df_check["next_id_session"] = df_check.groupby("choice_set")["id_session"].shift(-1)

df_check["same_as_prev"] = df_check["id_session"].eq(df_check["prev_id_session"])
df_check["same_as_next"] = df_check["id_session"].eq(df_check["next_id_session"])

consecutive_same_session = df_check[
    df_check["same_as_prev"] | df_check["same_as_next"]
]

consecutive_same_session[
    ["choice_set", "end_date", "id_session", "chosen", "same_as_prev", "same_as_next"]
].sort_values(["choice_set", "end_date"])


df_check["choice_set_count"] = (
    df_check.groupby("choice_set")["choice_set"].transform("size")
)

consecutive_same_session = df_check[
    (df_check["same_as_prev"] | df_check["same_as_next"])
]

consecutive_same_session[
    ["choice_set", "choice_set_count", "end_date", "id_session", "chosen"]
].sort_values(["choice_set", "end_date"])

,choice_set,choice_set_count,end_date,id_session,chosen


In [8]:
df_after_stage1[df_after_stage1['event_id'] == 254374]

,id_site,id_session,id_visitor,id_rep,start_date,start_time,end_date,end_time,duration,number_words,number_chars,number_lines,answer_canned,sentiment,accept_date,accept_time,read_date,read_time,event_type,event_type_desc,outcome,outcome_desc,subsession,delay,id_rep_code,sentiment_type,id_agent,id_agent_code,source_file,event_id
917175,1,100009301,200036222,30000199,11/05/2017 17:52:44,1494525164,11/05/2017 17:52:44,1494525164,0,80,380,0,3,0,11/05/2017 17:52:47,1494525167,11/05/2017 17:53:05,1494525185,1,visitor_line,1,served,1,0,2,3,30000199,200,D11052017,254374


In [9]:
df_reg_before[df_reg_before['choice_set'].isin(consecutive_same_session['choice_set'])][['choice_set', 'end_date', 'id_session', 'chosen', 'event_id']].sort_values(by = ['choice_set', 'end_date'], ascending=True)

,choice_set,end_date,id_session,chosen,event_id
38975,26907,13/05/2017 23:49:44,100093226,0,2485513
38976,26907,13/05/2017 23:49:44,100093226,0,2485514
38974,26907,13/05/2017 23:50:22,100087442,1,2361601
38977,26908,13/05/2017 23:49:44,100093226,1,2485513
38978,26908,13/05/2017 23:49:44,100093226,1,2485514
...,...,...,...,...,...
2299757,1594445,19/05/2017 23:48:45,100133132,1,3346068
2347438,1631825,20/05/2017 17:24:21,100056874,0,1552633
2347441,1631825,20/05/2017 17:25:32,100288852,0,4989177
2347439,1631825,20/05/2017 17:32:53,100250095,1,4745882


In [10]:
df_reg_before_filt = df_reg_before[~df_reg_before['choice_set'].isin(consecutive_same_session['choice_set'])]

In [11]:
df_reg_before_filt['n_messages'] = 1

C:\Users\Tamir\AppData\Local\Temp\ipykernel_34436\3898180209.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reg_before_filt['n_messages'] = 1


In [12]:
# df_reg_before_filt.to_csv('df_reg_filt_test_27_5.csv', index = 0)

In [13]:
# Checking alternative choice_set for same id_session:
df_reg_before[(df_reg_before['id_session'] == 100144015) & (df_reg_before['chosen'] == 1)]

,id_site,id_session,id_visitor,id_rep,start_date,start_time,end_date,end_time,duration,number_words,number_chars,number_lines,answer_canned,sentiment,accept_date,accept_time,read_date,read_time,event_type,event_type_desc,outcome,outcome_desc,subsession,delay,id_rep_code,sentiment_type,id_agent,id_agent_code,source_file,event_id,choice_set,chosen,waiting_time,workload,chosen_time,chosen_date
1545662,1,100144015,200077098,30000683,28/05/2017 00:23:48,1495931028,28/05/2017 00:23:52,1495931032,0,14,64,0,3,0,01/01/1970,0,01/01/1970,0,1,visitor_line,1,served,1,0,2,3,30000683,684,D28052017,3539224,1079591,1,138,13.0,1495931170,2017-05-28 00:26:10
1545667,1,100144015,200077098,30000683,28/05/2017 00:26:43,1495931203,28/05/2017 00:27:33,1495931253,50,2,9,0,2,1,28/05/2017 00:26:10,1495931170,28/05/2017 00:26:10,1495931170,1,visitor_line,1,served,1,0,684,1,30000683,684,D28052017,3539227,1079594,1,67,8.0,1495931320,2017-05-28 00:28:40
1545677,1,100144015,200077098,30000683,28/05/2017 01:14:50,1495934090,28/05/2017 01:19:39,1495934379,289,13,77,0,2,1,28/05/2017 01:16:53,1495934213,28/05/2017 01:18:10,1495934290,1,visitor_line,1,served,1,0,684,1,30000683,684,D28052017,3539230,1079602,1,723,9.0,1495935102,2017-05-28 01:31:42
1545680,1,100144015,200077098,30000683,28/05/2017 01:47:41,1495936061,28/05/2017 01:48:21,1495936101,40,7,31,0,2,0,28/05/2017 01:33:37,1495935217,28/05/2017 01:33:37,1495935217,1,visitor_line,1,served,1,0,684,3,30000683,684,D28052017,3539233,1079604,1,37,10.0,1495936138,2017-05-28 01:48:58
1545681,1,100144015,200077098,30000683,28/05/2017 01:48:58,1495936138,28/05/2017 01:50:21,1495936221,83,8,39,0,2,0,28/05/2017 01:48:58,1495936138,28/05/2017 01:48:58,1495936138,1,visitor_line,1,served,1,0,684,3,30000683,684,D28052017,3539235,1079605,1,23,10.0,1495936244,2017-05-28 01:50:44
1545682,1,100144015,200077098,30000683,28/05/2017 01:50:44,1495936244,28/05/2017 01:51:45,1495936305,61,14,79,0,2,1,28/05/2017 01:50:45,1495936245,28/05/2017 01:50:45,1495936245,1,visitor_line,1,served,1,0,684,1,30000683,684,D28052017,3539237,1079606,1,178,10.0,1495936483,2017-05-28 01:54:43
1545683,1,100144015,200077098,30000683,28/05/2017 01:56:20,1495936580,28/05/2017 02:01:07,1495936867,287,11,63,0,2,1,28/05/2017 01:54:43,1495936483,28/05/2017 01:54:43,1495936483,1,visitor_line,1,served,1,0,684,1,30000683,684,D28052017,3539239,1079607,1,64,10.0,1495936931,2017-05-28 02:02:11
1545684,1,100144015,200077098,30000683,28/05/2017 02:02:11,1495936931,28/05/2017 02:03:01,1495936981,50,7,55,0,2,1,28/05/2017 02:02:11,1495936931,28/05/2017 02:02:11,1495936931,1,visitor_line,1,served,1,0,684,1,30000683,684,D28052017,3539241,1079608,1,59,10.0,1495937040,2017-05-28 02:04:00
1545690,1,100144015,200077098,30000683,28/05/2017 02:22:52,1495938172,28/05/2017 02:36:23,1495938983,811,10,48,0,2,0,28/05/2017 02:29:59,1495938599,28/05/2017 02:29:59,1495938599,1,visitor_line,1,served,1,0,684,3,30000683,684,D28052017,3539244,1079614,1,41,9.0,1495939024,2017-05-28 02:37:04
1545691,1,100144015,200077098,30000683,28/05/2017 02:40:36,1495939236,28/05/2017 02:47:31,1495939651,415,7,34,0,2,0,28/05/2017 02:46:32,1495939592,28/05/2017 02:46:32,1495939592,1,visitor_line,1,served,1,0,684,3,30000683,684,D28052017,3539247,1079615,1,102,9.0,1495939753,2017-05-28 02:49:13


In [14]:
df_reg_before[df_reg_before['choice_set'] == 583210]

,id_site,id_session,id_visitor,id_rep,start_date,start_time,end_date,end_time,duration,number_words,number_chars,number_lines,answer_canned,sentiment,accept_date,accept_time,read_date,read_time,event_type,event_type_desc,outcome,outcome_desc,subsession,delay,id_rep_code,sentiment_type,id_agent,id_agent_code,source_file,event_id,choice_set,chosen,waiting_time,workload,chosen_time,chosen_date
824468,1,100118905,200180687,30000332,31/05/2017 20:42:59,1496263379,31/05/2017 20:46:35,1496263595,216,15,72,0,2,3,31/05/2017 20:36:35,1496262995,31/05/2017 20:38:26,1496263106,1,visitor_line,1,served,1,0,333,1,30000332,333,D31052017,3040819,583210,0,7,11.0,1496263602,2017-05-31 20:46:42
824469,1,100250663,200036454,30000332,31/05/2017 20:45:45,1496263545,31/05/2017 20:46:03,1496263563,8,13,63,0,3,0,01/01/1970,0,01/01/1970,0,1,visitor_line,1,served,1,0,2,3,30000332,333,D31052017,4749629,583210,1,39,11.0,1496263602,2017-05-31 20:46:42


In [15]:
# Analyzing relevant id_session in relevant dfs
df_exploded[(df_exploded['session_id_chosen'] == 100209964) & (df_exploded['time'] == 1495936117)]

,index,id_rep,event_id,time,session_id_chosen,concurrent_sessions,workload,chosen
7141154,1079603,30000683,4403901,1495936117,100209964,100056201,10.0,0
7141155,1079603,30000683,4403901,1495936117,100209964,100080524,10.0,0
7141156,1079603,30000683,4403901,1495936117,100209964,100235790,10.0,0
7141157,1079603,30000683,4403901,1495936117,100209964,100119101,10.0,0
7141158,1079603,30000683,4403901,1495936117,100209964,100144015,10.0,0
7141159,1079603,30000683,4403901,1495936117,100209964,100199220,10.0,0
7141160,1079603,30000683,4403901,1495936117,100209964,100209964,10.0,1
7141161,1079603,30000683,4403901,1495936117,100209964,100274601,10.0,0
7141162,1079603,30000683,4403901,1495936117,100209964,100322858,10.0,0
7141163,1079603,30000683,4403901,1495936117,100209964,100333979,10.0,0


Example of mismatch in chosen id_session and relevant id_sessions included in choice_set (fails 1 & 3 in 2nd function):

In [16]:
df_reg_before[df_reg_before['choice_set'] == 583210][['id_session', 'event_type', 'end_time', 'id_rep', 'chosen_time', 'chosen']]

,id_session,event_type,end_time,id_rep,chosen_time,chosen
824468,100118905,1,1496263595,30000332,1496263602,0
824469,100250663,1,1496263563,30000332,1496263602,1


In [17]:
df_old[(df_old[' end_time'] > 1496263350) & (df_old[' end_time'] <= 1496263602) & (df_old[' id_rep'] == 30000332)][[' id_session', ' event_type', ' end_time', ' id_rep']]

,id_session,event_type,end_time,id_rep
6521940,100118905,8,1496263379,30000332
6521941,100118905,1,1496263595,30000332
6596477,100250663,9,1496263595,30000332
6596478,100250663,2,1496263602,30000332
6596657,100251275,2,1496263379,30000332


In [18]:
df_after_stage1[(df_after_stage1['end_time'] > 1496263350) & (df_after_stage1['end_time'] <= 1496263602) & (df_after_stage1['id_rep'] == 30000332)][['id_session', 'event_type', 'end_time', 'id_rep']]

,id_session,event_type,end_time,id_rep
1387263,100251275,2,1496263379,30000332
1387264,100130219,1,1496263443,30000332
1387265,100250663,1,1496263553,30000332
1387266,100250663,7,1496263563,30000332
1387267,100118905,1,1496263595,30000332
1387268,100250663,2,1496263602,30000332


No chosen finding

In [15]:
choice_sets_without_chosen = (
    df_reg_before
    .groupby("choice_set")["chosen"]
    .sum()
)

choice_sets_without_chosen = choice_sets_without_chosen[
    choice_sets_without_chosen == 0
].index

df_no_chosen = df_reg_before[
    df_reg_before["choice_set"].isin(choice_sets_without_chosen)
].copy()


df_no_chosen["choice_set_count"] = (
    df_no_chosen.groupby("choice_set")["choice_set"].transform("size")
)

# df_no_chosen.sort_values(["choice_set", "end_date"])[['choice_set', 'id_session', 'event_type', 'end_time', 'id_rep', 'chosen_time', 'chosen']]
df_no_chosen.sort_values(["choice_set", "end_date"])[['choice_set', 'id_session', 'event_type', 'end_time', 'id_rep', 'chosen_time', 'chosen', 'size']]

,choice_set,id_session,event_type,end_time,id_rep,chosen_time,chosen,size
850,635,100297660,1,1493867335,30000002,1493867422,0,1
1389,1060,100295160,1,1494462611,1,1494462996,0,1
1390,1061,100295160,1,1494462611,1,1494463038,0,1
1393,1063,100295160,1,1494462611,1,1494463271,0,1
1738,1319,100150302,1,1494810928,30000002,1494810942,0,1
...,...,...,...,...,...,...,...,...
2338165,1633513,100258722,1,1495884479,30001400,1495884483,0,1
2338744,1634041,100271565,1,1495538965,30001401,1495539424,0,1
2339715,1634829,100056882,1,1495299511,30001426,1495299809,0,1
2340729,1635677,100103306,1,1495804449,30001427,1495804498,0,1


In [23]:
cols = ['choice_set', 'id_session', 'event_type', 'end_time', 'id_rep', 'chosen_time', 'chosen', 'size']

In [16]:
df_no_chosen['size'].value_counts()

size
1    3563
2    1582
3     279
4      36
6       6
5       5
Name: count, dtype: int64

In [17]:
df_no_chosen[df_no_chosen['id_rep'] == 1]['size'].value_counts()

size
1    37
2    21
3     7
4     1
Name: count, dtype: int64

In [18]:
df_no_chosen[(df_no_chosen['id_rep'] == 1) & (df_no_chosen['size'] == 2)]['choice_set']

204683      148422
228007      165292
794313      563999
947198      671971
947199      671971
1133089     802110
1147633     812240
1302547     920116
1359921     958981
1402342     988069
1503563    1056393
1535869    1078118
1580563    1108170
1761223    1232660
1812125    1265455
2028355    1414734
2080754    1450125
2111621    1471548
2132754    1486716
2164085    1507777
2317508    1618177
Name: choice_set, dtype: int64

In [24]:
df_no_chosen[df_no_chosen['choice_set'] == 148422][cols]

,choice_set,id_session,event_type,end_time,id_rep,chosen_time,chosen,size
204682,148422,100326149,1,1493828470,30000070,1493828477,0,2
204683,148422,100329479,1,1493828407,1,1493828477,0,2


In [36]:
df_old[(df_old[' id_session'] == 100329479)][[' id_session', ' event_type', ' end_time', ' id_rep', ' outcome_desc']]

,id_session,event_type,end_time,id_rep,outcome_desc
703596,100329479,1,1493828407,1,silent abandonment


In [46]:
df_old_silent = df_old[df_old[' outcome_desc'] == ' silent abandonment']
df_old_outcome_size = df_old_silent.groupby([' id_session']).size().reset_index(name = 'size')
df_old_outcome_size['size'].value_counts()

size
6     11650
4     11640
5     11498
7      8872
8      7352
3      6695
1      5599
9      5078
10     3784
11     2724
12     1789
13     1179
14      859
2       544
15      510
16      343
17      234
18      150
19       98
20       60
21       36
22       30
24       17
23       16
25       12
26       11
27        8
29        3
35        3
30        3
33        2
34        2
31        2
28        2
Name: count, dtype: int64

In [7]:
df_old[' outcome_desc'].value_counts()

 outcome_desc
served                   6066772
silent abandonment        510375
known abandonment          19430
served and transfered      12346
Unknown                     5207
Name: count, dtype: int64

In [ ]:
#######

In [16]:
df_old = pd.read_csv('merged_session_events.csv')

In [21]:
df_old[df_old[' id_session'] == 100057841][[' id_session', ' event_type', ' end_time', ' end_date', ' id_rep', ' outcome_desc', ' subsession']].sort_values(by = [' id_session', ' end_time'], ascending=True).head(30)

,id_session,event_type,end_time,end_date,id_rep,outcome_desc,subsession
6458105,100057841,2,1496272372,31/05/2017 23:12:52,30000546,known abandonment,1
6458106,100057841,8,1496272413,31/05/2017 23:13:33,30000546,known abandonment,1
6458107,100057841,1,1496272433,31/05/2017 23:13:53,30000546,known abandonment,1
6458108,100057841,9,1496272454,31/05/2017 23:14:14,30000546,known abandonment,1
6458109,100057841,2,1496272467,31/05/2017 23:14:27,30000546,known abandonment,1
6458110,100057841,8,1496273560,31/05/2017 23:32:40,30000546,known abandonment,1
6458111,100057841,2,1496273722,31/05/2017 23:35:22,30000546,known abandonment,1
6458112,100057841,2,1496273792,31/05/2017 23:36:32,30000546,known abandonment,1
6458113,100057841,1,1496273841,31/05/2017 23:37:21,30000546,known abandonment,1
6458114,100057841,2,1496273984,31/05/2017 23:39:44,30000546,known abandonment,1


In [22]:
df_old[' outcome_desc'].value_counts()

 outcome_desc
served                   6066772
silent abandonment        510375
known abandonment          19430
served and transfered      12346
Unknown                     5207
Name: count, dtype: int64

In [24]:
df_filt_1 = df_old[(df_old[' outcome_desc'] == ' known abandonment')][[' id_session', ' event_type', ' end_time', ' id_rep', ' outcome_desc']].sort_values(by = [' id_session', ' end_time'], ascending=True)

df_filt_1[' id_session'].nunique()

16735

In [23]:
df_filt = df_old[(df_old[' outcome_desc'] == ' known abandonment') & (df_old[' event_type'] == 2)][[' id_session', ' event_type', ' end_time', ' id_rep', ' outcome_desc']].sort_values(by = [' id_session', ' end_time'], ascending=True)

df_filt[' id_session'].nunique()

30

In [ ]:
df_old_session_size = df_old.groupby([' id_session']).size().reset_index(name = 'size')
df_old = df_old.merge(df_old_session_size, on = ' id_session', how = 'left')
df_old_size_11 = df_old[df_old['size_x'] > 5]
df_old_filt = df_old_size_11[df_old_size_11[' outcome_desc'] == ' known abandonment']

--------------------------

In [15]:
df_old_filt[[' id_session', ' event_type', ' end_time', ' id_rep', ' outcome_desc', 'size_x']].sort_values(by = [' id_session', ' end_time'], ascending=True).head(30)

,id_session,event_type,end_time,id_rep,outcome_desc,size_x
6458105,100057841,2,1496272372,30000546,known abandonment,19
6458106,100057841,8,1496272413,30000546,known abandonment,19
6458107,100057841,1,1496272433,30000546,known abandonment,19
6458108,100057841,9,1496272454,30000546,known abandonment,19
6458109,100057841,2,1496272467,30000546,known abandonment,19
6458110,100057841,8,1496273560,30000546,known abandonment,19
6458111,100057841,2,1496273722,30000546,known abandonment,19
6458112,100057841,2,1496273792,30000546,known abandonment,19
6458113,100057841,1,1496273841,30000546,known abandonment,19
6458114,100057841,2,1496273984,30000546,known abandonment,19


In [14]:
df_old[' outcome_desc'].value_counts()

 outcome_desc
served                   6066772
silent abandonment        510375
known abandonment          19430
served and transfered      12346
Unknown                     5207
Name: count, dtype: int64

In [9]:
df_old_filt[' event_type'].value_counts()

 event_type
2    196781
8     61809
1     50498
7     44581
9     25884
Name: count, dtype: int64

In [56]:
df_old[df_old[' id_session'] == 100161819][[' id_session', ' event_type', ' end_time', ' id_rep', ' outcome_desc']]

,id_session,event_type,end_time,id_rep,outcome_desc
173773,100161819,1,1493603467,1,silent abandonment
173774,100161819,7,1493603706,1,silent abandonment
173775,100161819,2,1493603717,30000023,silent abandonment
173776,100161819,2,1493603730,30000023,silent abandonment
173777,100161819,2,1493603736,30000023,silent abandonment
173778,100161819,2,1493603739,30000023,silent abandonment
173779,100161819,2,1493603750,30000023,silent abandonment
173780,100161819,2,1493603758,30000023,silent abandonment
173781,100161819,8,1493609524,30000023,silent abandonment
173782,100161819,2,1493613981,30000023,silent abandonment


In [45]:
df_old_silent

,id_site,id_session,id_visitor,id_rep,start_date,start_time,end_date,end_time,duration,number_words,number_chars,number_lines,answer_canned,sentiment,accept_date,accept_time,read_date,read_time,event_type,event_type_desc,outcome,outcome_desc,subsession,delay,id_rep_code,sentiment_type,id_agent,id_agent_code,source_file,event_id


In [30]:
df_old[' outcome_desc'].value_counts()

 outcome_desc
served                   6066772
silent abandonment        510375
known abandonment          19430
served and transfered      12346
Unknown                     5207
Name: count, dtype: int64

In [34]:
df_old[df_old[' outcome_desc'] == ' silent abandonment'][' id_visitor'].nunique()

70266

In [31]:
pd.crosstab(df_old[' outcome'], df_old[' outcome_desc'])

outcome_desc,Unknown,known abandonment,served,served and transfered,silent abandonment
outcome,,,,,
1,0,0,6066772,0,0
2,0,0,0,12346,0
3,0,0,0,0,510375
4,0,19430,0,0,0
7,5207,0,0,0,0


In [29]:
df_old[df_old[' id_session'] == 100329479][' outcome_desc']

703596     silent abandonment
Name:  outcome_desc, dtype: str

In [42]:
df_reg_before.shape

(2342263, 32)

In [29]:
df_after_stage1[(df_after_stage1['end_time'] >= 1494722786) & (df_after_stage1['end_time'] <= 1494723128) & (df_after_stage1['id_rep'] == 30000005)][['id_session', 'event_type', 'end_time', 'id_rep']]

,id_session,event_type,end_time,id_rep
21669,100007537,1,1494722786,30000005
21670,100078472,1,1494722947,30000005
21671,100078472,1,1494722956,30000005
21672,100078472,2,1494722970,30000005
21673,100257109,2,1494722998,30000005
21674,100078472,1,1494722999,30000005
21675,100078472,1,1494723013,30000005
21676,100078472,2,1494723013,30000005
21677,100072907,2,1494723032,30000005
21678,100072907,1,1494723096,30000005


In [39]:
df_after_stage1[(df_after_stage1['id_session'] == 100078472) & (df_after_stage1['end_time'] <= 1494723128)][['id_session', 'event_type', 'end_time', 'id_rep']].sort_values(by = ['end_time', 'event_type'], ascending= [True, False])

,id_session,event_type,end_time,id_rep
21640,100078472,1,1494721613,30000005
21647,100078472,7,1494721858,30000005
21648,100078472,2,1494722007,30000005
21649,100078472,1,1494722127,30000005
21652,100078472,2,1494722260,30000005
21655,100078472,1,1494722444,30000005
21656,100078472,2,1494722447,30000005
21657,100078472,1,1494722458,30000005
21658,100078472,2,1494722461,30000005
21659,100078472,1,1494722471,30000005


In [37]:
df_old[(df_old[' id_session'] == 100078472) & (df_old[' end_time'] <= 1494723128)][[' id_session', ' event_type', ' end_time', ' id_rep']].sort_values(by = ' end_time')

,id_session,event_type,end_time,id_rep
2859403,100078472,1,1494721613,1
2859404,100078472,7,1494721858,1
2859405,100078472,9,1494721990,30000005
2859406,100078472,2,1494722007,30000005
2859407,100078472,2,1494722008,30000005
2859408,100078472,2,1494722044,30000005
2859409,100078472,2,1494722048,30000005
2859410,100078472,1,1494722127,30000005
2859411,100078472,9,1494722233,30000005
2859412,100078472,2,1494722260,30000005


In [8]:
df_no_chosen['choice_set'].nunique()

4458

In [27]:
df_no_chosen[df_no_chosen['choice_set_count'] > 1].sort_values(["choice_set", "end_date"])[['choice_set', 'id_session', 'event_type', 'end_time', 'id_rep', 'chosen_time', 'chosen']]

,choice_set,id_session,event_type,end_time,id_rep,chosen_time,chosen
5504,4207,100115393,1,1494619233,30000004,1494619827,0
5503,4207,100100676,1,1494619375,30000004,1494619827,0
7734,5600,100086285,1,1495726846,30000004,1495727044,0
7735,5600,100155079,1,1495727013,30000004,1495727044,0
8745,6305,100014761,1,1493684210,30000005,1493684920,0
...,...,...,...,...,...,...,...
2323752,1622175,100061900,1,1496237148,30001377,1496237260,0
2333001,1629438,100202452,1,1494949286,30001394,1494953452,0
2333000,1629438,100133658,1,1494953445,30001394,1494953452,0
2333736,1629931,100072119,1,1495382025,30001394,1495382286,0


In [8]:
df_reg_before[df_reg_before['choice_set'] == 58]

,id_site,id_session,id_visitor,id_rep,start_date,start_time,end_date,end_time,duration,number_words,number_chars,number_lines,answer_canned,sentiment,event_type,event_type_desc,outcome,outcome_desc,subsession,id_rep_code,sentiment_type,id_agent,id_agent_code,source_file,event_id,n_messages,choice_set,chosen,waiting_time,workload,chosen_time,chosen_date
85,1,100132142,200026922,30000002,01/05/2017 02:37:11,1493606231,01/05/2017 02:42:06,1493606526,0,5,23,0,3,0,1,visitor_line,1,served,1,2,3,30000002,3,D01052017,3324791,1,58,1,115,7.0,1493606641,2017-05-01 02:44:01
86,1,100272337,200170576,30000002,01/05/2017 02:39:20,1493606360,01/05/2017 02:39:20,1493606360,41,19,78,0,3,0,1,visitor_line,1,served,1,2,3,30000002,3,D01052017,4892830,2,58,0,281,7.0,1493606641,2017-05-01 02:44:01


In [23]:
df_old[(df_old[' start_time'] >= 1493606517) & (df_old[' end_time'] <= 1493606641) & (df_old[' id_rep'] == 30000002)][[' id_session', ' event_type', ' end_time', ' id_rep']]

,id_session,event_type,end_time,id_rep
151169,100132142,9,1493606558,30000002
151170,100132142,2,1493606641,30000002
221679,100272337,1,1493606558,30000002
221680,100272337,7,1493606558,30000002


In [24]:
df_old[(df_old[' id_session'] == 100132142)][[' id_session', ' event_type', ' end_time', ' id_rep']]

,id_session,event_type,end_time,id_rep
151167,100132142,1,1493606231,1
151168,100132142,7,1493606526,1
151169,100132142,9,1493606558,30000002
151170,100132142,2,1493606641,30000002
151171,100132142,2,1493606644,30000002
151172,100132142,8,1493606725,30000002
151173,100132142,1,1493606760,30000002
151174,100132142,2,1493606795,30000002
151175,100132142,1,1493606808,30000002
151176,100132142,2,1493606845,30000002


In [25]:
df_after_stage1[(df_after_stage1['id_session'] == 100132142)][['id_session', 'event_type', 'end_time', 'id_rep']]

,id_session,event_type,end_time,id_rep
4835,100132142,1,1493606231,30000002
4847,100132142,7,1493606526,30000002
4850,100132142,2,1493606641,30000002
4852,100132142,1,1493606760,30000002
4853,100132142,2,1493606795,30000002
4854,100132142,1,1493606808,30000002
4855,100132142,2,1493606845,30000002
4856,100132142,1,1493607032,30000002
4857,100132142,2,1493607228,30000002
4858,100132142,1,1493607278,30000002


In [26]:
len(df_no_chosen[df_no_chosen['choice_set_count'] > 1]['choice_set'].unique())

832

In [27]:
df_old[(df_old[' id_session'] == 100167489) & (df_old[' end_time'] == 1495728863)]

,id_site,id_session,id_visitor,id_rep,start_date,start_time,end_date,end_time,duration,number_words,number_chars,number_lines,answer_canned,sentiment,accept_date,accept_time,read_date,read_time,event_type,event_type_desc,outcome,outcome_desc,subsession,delay,id_rep_code,sentiment_type,id_agent,id_agent_code,source_file
5427717,1,100167489,200019210,1,25/05/2017 16:13:48,1495728828,25/05/2017 16:14:23,1495728863,35,51,232,0,3,0,01/01/1970,0,01/01/1970,0,1,visitor_line,1,served,1,0,2,3,30000708,709,D25052017


In [28]:
# Check: are there ever multiple type=7 events between a type=1 and the next type=2?
# If a type=7 is immediately preceded by another type=7 within the same session, that's a case we need to handle.
df_ordered = merged_df[merged_df['event_type'].isin([1, 2, 7])].sort_values(['id_session', 'end_time']).copy()
df_ordered['prev_event_type'] = df_ordered.groupby('id_session')['event_type'].shift(1)

consecutive_7 = df_ordered[(df_ordered['event_type'] == 7) & (df_ordered['prev_event_type'] == 7)]
print(f"Consecutive type=7 events (type=7 immediately following type=7): {len(consecutive_7):,}")
consecutive_7[['id_session', 'event_type', 'end_time', 'prev_event_type']].head(10)

Consecutive type=7 events (type=7 immediately following type=7): 0


,id_session,event_type,end_time,prev_event_type


In [29]:
# Transform: for each event_type=1 whose immediate next event in the same session is event_type=7,
# update its end_time and end_date to those of the type=7 event.
df_merged_fixed = merged_df.sort_values(['id_session', 'end_time']).reset_index(drop=True).copy()

df_merged_fixed['_next_event_type'] = df_merged_fixed.groupby('id_session')['event_type'].shift(-1)
df_merged_fixed['_next_end_time'] = df_merged_fixed.groupby('id_session')['end_time'].shift(-1)
df_merged_fixed['_next_end_date'] = df_merged_fixed.groupby('id_session')['end_date'].shift(-1)

mask = (df_merged_fixed['event_type'] == 1) & (df_merged_fixed['_next_event_type'] == 7)
df_merged_fixed.loc[mask, 'end_time'] = df_merged_fixed.loc[mask, '_next_end_time']
df_merged_fixed.loc[mask, 'end_date'] = df_merged_fixed.loc[mask, '_next_end_date']

df_merged_fixed = df_merged_fixed.drop(columns=['_next_event_type', '_next_end_time', '_next_end_date'])

print(f"Rows with end_time/end_date updated to type=7 values: {mask.sum():,}")

Rows with end_time/end_date updated to type=7 values: 266,371


In [30]:
before = len(df_merged_fixed)
df_merged_fixed = df_merged_fixed[df_merged_fixed['event_type'] != 7].reset_index(drop=True)
print(f"Dropped {before - len(df_merged_fixed):,} type=7 rows. Remaining: {len(df_merged_fixed):,}")

Dropped 266,809 type=7 rows. Remaining: 3,622,578


In [31]:
merged_df[merged_df['id_session'] == 100169954][['event_type', 'end_time', 'end_date']]

,event_type,end_time,end_date
1251252,1,1495429655,22/05/2017 05:07:35
1251253,7,1495429658,22/05/2017 05:07:38
1251255,2,1495429702,22/05/2017 05:08:22


In [32]:
df_merged_fixed[df_merged_fixed['id_session'] == 100169954][['event_type', 'end_time', 'end_date']]

,event_type,end_time,end_date
2873584,1,1495429658,22/05/2017 05:07:38
2873585,2,1495429702,22/05/2017 05:08:22


In [33]:
df_merged_fixed.to_csv('agents_merged_fixed_28_5.csv', index = 0)

## No-chosen choice sets — root cause analysis (post Stage 3 rerun)

In [6]:
# 1. Size distribution: single-alternative vs multi-alternative no-chosen sets
size_dist = df_no_chosen['choice_set_count'].value_counts().sort_index()
print("Choice set size distribution among no-chosen sets:")
print(size_dist)
print(f"\nSingle-alternative (size=1): {(df_no_chosen['choice_set_count'] == 1).sum():,} rows "
      f"({df_no_chosen[df_no_chosen['choice_set_count'] == 1]['choice_set'].nunique():,} choice sets)")
print(f"Multi-alternative  (size>1): {(df_no_chosen['choice_set_count'] > 1).sum():,} rows "
      f"({df_no_chosen[df_no_chosen['choice_set_count'] > 1]['choice_set'].nunique():,} choice sets)")

NameError: name 'df_no_chosen' is not defined

In [10]:
# 2. id_rep=1 check: sessions where the rep_id fix found no valid agent (still id_rep=1)
no_chosen_rep1 = df_no_chosen[df_no_chosen['id_rep'] == 1]
print(f"No-chosen rows with id_rep=1: {len(no_chosen_rep1):,} "
      f"({no_chosen_rep1['choice_set'].nunique():,} choice sets)")
print(f"As % of all no-chosen choice sets: "
      f"{no_chosen_rep1['choice_set'].nunique() / df_no_chosen['choice_set'].nunique() * 100:.1f}%")

No-chosen rows with id_rep=1: 66 (64 choice sets)
As % of all no-chosen choice sets: 1.4%


In [11]:
# 3. Agent concentration: which id_rep values drive the most no-chosen sets
rep_counts = (
    df_no_chosen.groupby('id_rep')['choice_set']
    .nunique()
    .sort_values(ascending=False)
)
print("Top 15 agents by no-chosen choice set count:")
print(rep_counts.head(15))
print(f"\nTotal agents with at least one no-chosen set: {len(rep_counts):,}")

Top 15 agents by no-chosen choice set count:
id_rep
1           64
30000700    44
30000149    32
30000402    28
30000035    27
30000014    26
30000554    25
30000065    24
30000077    22
30000395    22
30000524    21
30000500    21
30000359    21
30000205    20
30000133    20
Name: choice_set, dtype: int64

Total agents with at least one no-chosen set: 767


In [12]:
# 4. Root cause: for multi-alternative no-chosen sets, is the chosen session 
# from df_exploded actually present in df_reg_before for that choice set?
# If not -> Stage 3 never produced a row for the chosen session (no valid type=1 found)
# If yes -> it's there but incorrectly has chosen=0

multi_no_chosen_sets = df_no_chosen[df_no_chosen['choice_set_count'] > 1]['choice_set'].unique()

# Get the expected chosen session for each of these choice sets from df_exploded
exploded_chosen = df_exploded[
    (df_exploded['chosen'] == 1) &
    (df_exploded['index'].isin(multi_no_chosen_sets))
][['index', 'session_id_chosen']].drop_duplicates()
exploded_chosen.columns = ['choice_set', 'expected_chosen_session']

# Check which of those appear in df_reg_before for the same choice set
reg_sessions = df_reg_before[['choice_set', 'id_session']].drop_duplicates()
merged_check = exploded_chosen.merge(reg_sessions, on='choice_set', how='left')
merged_check['chosen_session_present'] = (
    merged_check['expected_chosen_session'] == merged_check['id_session']
)
present = merged_check.groupby('choice_set')['chosen_session_present'].any()

print(f"Multi-alternative no-chosen sets: {len(multi_no_chosen_sets):,}")
print(f"  Chosen session absent from choice set (Stage 3 couldn't find type=1): "
      f"{(~present).sum():,} ({(~present).mean()*100:.1f}%)")
print(f"  Chosen session present but marked chosen=0 (logic error):              "
      f"{present.sum():,} ({present.mean()*100:.1f}%)")

Multi-alternative no-chosen sets: 895
  Chosen session absent from choice set (Stage 3 couldn't find type=1): 676 (100.0%)
  Chosen session present but marked chosen=0 (logic error):              0 (0.0%)


In [ ]:
df_choice = pd.read_csv('df_choicesets_28_05_2026.csv')
df_choice['chosen_time'] = df_choice['end_time'] + df_choice['waiting_time']
df_choice['chosen_date'] = pd.to_datetime(df_choice['chosen_time'], unit = 's')
df_choice

,id_site,id_session,id_visitor,id_rep,start_date,start_time,end_date,end_time,duration,number_words,number_chars,number_lines,answer_canned,sentiment,accept_date,accept_time,read_date,read_time,event_type,event_type_desc,outcome,outcome_desc,subsession,delay,id_rep_code,sentiment_type,id_agent,id_agent_code,source_file,event_id,choice_set,chosen,waiting_time,workload,size,chosen_time,chosen_date
0,1,100247275,200140251,30000002,01/05/2017 00:22:57,1493598177,01/05/2017 00:24:04,1493598244,0,36,164,0,3,0,01/01/1970,0,01/01/1970,0,1,visitor_line,1,served,1,0,2,3,30000002,3,D01052017,4726408,0,1,14,2.0,1.0,1493598258,2017-05-01 00:24:18
1,1,100247275,200140251,30000002,01/05/2017 00:24:29,1493598269,01/05/2017 00:25:11,1493598311,42,1,6,0,2,1,01/05/2017 00:24:30,1493598270,01/05/2017 00:24:30,1493598270,1,visitor_line,1,served,1,0,3,1,30000002,3,D01052017,4726413,1,1,234,2.0,1.0,1493598545,2017-05-01 00:29:05
2,1,100045597,200194746,30000002,01/05/2017 00:37:28,1493599048,01/05/2017 00:37:32,1493599052,0,2,7,0,3,0,01/01/1970,0,01/01/1970,0,1,visitor_line,1,served,1,0,2,3,30000002,3,D01052017,1247139,2,1,15,3.0,1.0,1493599067,2017-05-01 00:37:47
3,1,100045597,200194746,30000002,01/05/2017 00:37:47,1493599067,01/05/2017 00:39:04,1493599144,77,48,206,0,2,0,01/05/2017 00:37:47,1493599067,01/05/2017 00:37:47,1493599067,1,visitor_line,1,served,1,0,3,3,30000002,3,D01052017,1247142,3,1,38,3.0,1.0,1493599182,2017-05-01 00:39:42
4,1,100045597,200194746,30000002,01/05/2017 00:39:50,1493599190,01/05/2017 00:41:11,1493599271,81,7,31,0,2,0,01/05/2017 00:39:50,1493599190,01/05/2017 00:39:50,1493599190,1,visitor_line,1,served,1,0,3,3,30000002,3,D01052017,1247145,4,1,21,3.0,1.0,1493599292,2017-05-01 00:41:32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2353467,1,100144721,200208938,30001566,31/05/2017 22:41:10,1496270470,31/05/2017 22:41:10,1496270470,0,14,59,0,3,0,01/01/1970,0,01/01/1970,0,1,visitor_line,1,served,1,0,2,3,30001566,1567,D31052017,3550994,1636894,0,471,2.0,NaN,1496270941,2017-05-31 22:49:01
2353468,1,100034751,200021764,30001566,31/05/2017 22:49:01,1496270941,31/05/2017 22:49:07,1496270947,6,1,2,0,2,0,31/05/2017 22:49:02,1496270942,31/05/2017 22:49:02,1496270942,1,visitor_line,1,served,1,0,1567,3,30001566,1567,D31052017,953337,1636895,0,80,2.0,NaN,1496271027,2017-05-31 22:50:27
2353469,1,100144721,200208938,30001566,31/05/2017 22:49:07,1496270947,31/05/2017 22:49:29,1496270969,22,17,77,0,2,0,31/05/2017 22:41:16,1496270476,31/05/2017 22:42:02,1496270522,1,visitor_line,1,served,1,0,1567,3,30001566,1567,D31052017,3550995,1636895,1,58,2.0,NaN,1496271027,2017-05-31 22:50:27
2353470,1,100034751,200021764,30001566,31/05/2017 22:49:01,1496270941,31/05/2017 22:49:07,1496270947,6,1,2,0,2,0,31/05/2017 22:49:02,1496270942,31/05/2017 22:49:02,1496270942,1,visitor_line,1,served,1,0,1567,3,30001566,1567,D31052017,953337,1636896,1,144,2.0,NaN,1496271091,2017-05-31 22:51:31


In [64]:
grouped = df_choice.groupby('choice_set').size().reset_index(name = 'count_rows')
grouped[grouped['count_rows'] >= 2]

df_choice[df_choice['choice_set'] ==  26][['choice_set', 'id_session', 'id_rep', 'event_type', 'end_date', 'end_time', 'chosen', 'chosen_time']].sort_values(by = (['end_time']))

,choice_set,id_session,id_rep,event_type,end_date,end_time,chosen,chosen_time
34,26,100063972,30000002,1,01/05/2017 01:25:58,1493601958,1,1493602107
35,26,100151963,30000002,1,01/05/2017 01:27:20,1493602040,0,1493602107
33,26,100057443,30000002,1,01/05/2017 01:28:07,1493602087,0,1493602107


In [67]:
df_after_stage1[(df_after_stage1['end_time'] >= 1493601958) & (df_after_stage1['end_time'] <= 1493602107) & (df_after_stage1['id_rep'] == 30000002)][['id_session', 'event_type', 'end_time', 'id_rep']]

,id_session,event_type,end_time,id_rep
4769,100063972,1,1493601958,30000002
4770,100151963,2,1493601974,30000002
4771,100151963,1,1493602040,30000002
4772,100057443,1,1493602087,30000002
4773,100063972,2,1493602107,30000002


In [18]:
# Query df_reg_before for choice_sets with more than one event_type=1 for the same id_session
result = df_reg_before[
    df_reg_before['choice_set'].isin(multi_no_chosen_sets)
].groupby(['choice_set', 'id_session']).apply(
    lambda x: (x.duplicated(keep = False)).sum()
).reset_index(name='count_type_1')

# Filter for those with more than one type=1 event
multiple_type_1 = result[result['count_type_1'] > 1]

# Get full details for these rows
details = df_reg_before[
    (df_reg_before['choice_set'].isin(multiple_type_1['choice_set'])) &
    (df_reg_before['id_session'].isin(multiple_type_1['id_session'])) &
    (df_reg_before['event_type'] == 1)
][['choice_set', 'id_session', 'event_type', 'end_time', 'end_date', 'chosen', 'event_id']].sort_values(
    ['choice_set', 'id_session', 'end_time']
)

print(f"Choice sets with multiple type=1 events for same id_session: {len(multiple_type_1)}")
details.head(20)

Choice sets with multiple type=1 events for same id_session: 0


C:\Users\Tamir\AppData\Local\Temp\ipykernel_19624\1991381106.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ].groupby(['choice_set', 'id_session']).apply(


,choice_set,id_session,event_type,end_time,end_date,chosen,event_id


In [23]:
df_reg_before[df_reg_before['n_messages'] > 1][['choice_set', 'id_session', 'id_rep', 'n_messages', 'number_words', 'chosen', 'chosen_time']].sort_values('n_messages', ascending=False)

,choice_set,id_session,id_rep,n_messages,number_words,chosen,chosen_time
1034064,733934,100063760,30000448,19,781,0,1495895299
1034060,733932,100063760,30000448,19,781,0,1495895053
1034068,733936,100063760,30000448,19,781,0,1495895527
1034062,733933,100063760,30000448,19,781,0,1495895104
1034066,733935,100063760,30000448,19,781,0,1495895469
...,...,...,...,...,...,...,...
2342183,1636840,100132070,30001495,2,20,1,1494026051
2342184,1636841,100132070,30001495,2,2,1,1494026112
2342194,1636849,100045743,30001495,2,22,1,1494026584
2342240,1636884,100034751,30001566,2,24,1,1496269788


In [25]:
df_reg_before[df_reg_before['choice_set'] == 733934]

,id_site,id_session,id_visitor,id_rep,start_date,start_time,end_date,end_time,duration,number_words,number_chars,number_lines,answer_canned,sentiment,event_type,event_type_desc,outcome,outcome_desc,subsession,id_rep_code,sentiment_type,id_agent,id_agent_code,source_file,event_id,n_messages,choice_set,chosen,waiting_time,workload,chosen_time,chosen_date
1034064,1,100063760,200021853,30000448,27/05/2017 14:20:06,1495894806,27/05/2017 14:20:41,1495894841,153,781,4236,0,2,14,1,visitor_line,1,served,1,449,3,30000448,449,D27052017,1742137,19,733934,0,458,6.0,1495895299,2017-05-27 14:28:19
1034065,1,100081232,200038816,30000448,27/05/2017 14:27:07,1495895227,27/05/2017 14:28:01,1495895281,54,32,180,0,2,1,1,visitor_line,1,served,1,449,1,30000448,449,D27052017,2214734,1,733934,1,18,6.0,1495895299,2017-05-27 14:28:19


In [27]:
# Investigate n_messages=19: session 100063760, id_rep=30000448
# Look at the full event sequence in the cleaned data to verify 19 consecutive type=1s
df_after_stage1[df_after_stage1['id_session'] == 100063760][
    ['id_session', 'event_type', 'end_time', 'end_date', 'id_rep', 'number_words']
].sort_values('end_time')

,id_session,event_type,end_time,end_date,id_rep,number_words
1748352,100063760,1,1495893858,27/05/2017 14:04:18,30000448,1
1748353,100063760,7,1495893860,27/05/2017 14:04:20,30000448,0
1748354,100063760,2,1495893883,27/05/2017 14:04:43,30000448,30
1748355,100063760,1,1495893956,27/05/2017 14:05:56,30000448,58
1748358,100063760,2,1495894127,27/05/2017 14:08:47,30000448,43
1748359,100063760,1,1495894153,27/05/2017 14:09:13,30000448,11
1748360,100063760,2,1495894212,27/05/2017 14:10:12,30000448,31
1748365,100063760,1,1495894611,27/05/2017 14:16:51,30000448,194
1748368,100063760,1,1495894718,27/05/2017 14:18:38,30000448,31
1748369,100063760,2,1495894744,27/05/2017 14:19:04,30000448,10


In [3]:
df_reg_before

,id_site,id_session,id_visitor,id_rep,start_date,start_time,end_date,end_time,duration,number_words,number_chars,number_lines,answer_canned,sentiment,event_type,event_type_desc,outcome,outcome_desc,subsession,id_rep_code,sentiment_type,id_agent,id_agent_code,source_file,event_id,n_messages,choice_set,chosen,waiting_time,workload
0,1,100247275,200140251,30000002,01/05/2017 00:22:57,1493598177,01/05/2017 00:24:04,1493598244,0,36,164,0,3,0,1,visitor_line,1,served,1,2,3,30000002,3,D01052017,4726408,1,0,1,14,2.0
1,1,100247275,200140251,30000002,01/05/2017 00:24:29,1493598269,01/05/2017 00:25:11,1493598311,42,1,6,0,2,1,1,visitor_line,1,served,1,3,1,30000002,3,D01052017,4726413,1,1,1,234,2.0
2,1,100045597,200194746,30000002,01/05/2017 00:37:28,1493599048,01/05/2017 00:37:32,1493599052,0,2,7,0,3,0,1,visitor_line,1,served,1,2,3,30000002,3,D01052017,1247139,1,2,1,15,3.0
3,1,100045597,200194746,30000002,01/05/2017 00:37:47,1493599067,01/05/2017 00:39:04,1493599144,77,48,206,0,2,0,1,visitor_line,1,served,1,3,3,30000002,3,D01052017,1247142,1,3,1,38,3.0
4,1,100045597,200194746,30000002,01/05/2017 00:39:50,1493599190,01/05/2017 00:41:11,1493599271,81,7,31,0,2,0,1,visitor_line,1,served,1,3,3,30000002,3,D01052017,1247145,1,4,1,21,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2342258,1,100144721,200208938,30001566,31/05/2017 22:41:10,1496270470,31/05/2017 22:41:10,1496270470,0,14,59,0,3,0,1,visitor_line,1,served,1,2,3,30001566,1567,D31052017,3550994,1,1636894,0,471,2.0
2342259,1,100034751,200021764,30001566,31/05/2017 22:49:01,1496270941,31/05/2017 22:49:07,1496270947,6,1,2,0,2,0,1,visitor_line,1,served,1,1567,3,30001566,1567,D31052017,953337,1,1636895,0,80,2.0
2342260,1,100144721,200208938,30001566,31/05/2017 22:41:10,1496270470,31/05/2017 22:41:10,1496270470,22,31,136,0,3,0,1,visitor_line,1,served,1,2,3,30001566,1567,D31052017,3550994,2,1636895,1,557,2.0
2342261,1,100034751,200021764,30001566,31/05/2017 22:49:01,1496270941,31/05/2017 22:49:07,1496270947,6,1,2,0,2,0,1,visitor_line,1,served,1,1567,3,30001566,1567,D31052017,953337,1,1636896,1,144,2.0


In [4]:
df_reg_before[df_reg_before['choice_set'] == 350744][['id_session', 'event_type', 'end_time', 'id_rep', 'chosen_time', 'chosen']]

,id_session,event_type,end_time,id_rep,chosen_time,chosen
484948,100100136,1,1494682547,30000169,1494682566,1
484949,100159633,1,1494682556,30000169,1494682566,0


In [26]:
df_after_stage1[(df_after_stage1['end_time'] >= 1494270708) & (df_after_stage1['end_time'] <= 1494272345) & (df_after_stage1['id_rep'] == 30000143)][['id_session', 'event_type', 'end_time', 'id_rep']].sort_values(by = ['end_time', 'event_type'], ascending= [True, False]) 

,id_session,event_type,end_time,id_rep
1897645,100098539,1,1494270708,30000143
888036,100044445,1,1494270759,30000143
888037,100044445,2,1494270768,30000143
2380648,100129362,1,1494270857,30000143
2395327,100130361,1,1494270869,30000143
2010517,100105684,1,1494270986,30000143
2380649,100129362,2,1494271070,30000143
2395328,100130361,2,1494271082,30000143
2010518,100105684,2,1494271091,30000143
2111947,100112152,1,1494271123,30000143


In [24]:
df_reg_before[df_reg_before['choice_set'] == 302733][['id_session', 'event_type', 'end_time', 'id_rep', 'chosen_time', 'chosen', 'n_messages']].sort_values(by = ['end_time'])    

,id_session,event_type,end_time,id_rep,chosen_time,chosen,n_messages
417869,100098539,1,1494270708,30000143,1494272345,1,2


In [ ]:
df_exploded[df_exploded['time'] == 1494272345]

,index,id_rep,event_id,time,session_id_chosen,concurrent_sessions,workload,chosen
1861527,302733,30000143,2600290,1494272345,100098539,100030178,13.0,0
1861528,302733,30000143,2600290,1494272345,100098539,100098539,13.0,1
1861529,302733,30000143,2600290,1494272345,100098539,100105684,13.0,0
1861530,302733,30000143,2600290,1494272345,100098539,100112152,13.0,0
1861531,302733,30000143,2600290,1494272345,100098539,100129362,13.0,0
1861532,302733,30000143,2600290,1494272345,100098539,100185885,13.0,0
1861533,302733,30000143,2600290,1494272345,100098539,100229064,13.0,0
1861534,302733,30000143,2600290,1494272345,100098539,100257458,13.0,0
1861535,302733,30000143,2600290,1494272345,100098539,100279296,13.0,0
1861536,302733,30000143,2600290,1494272345,100098539,100281590,13.0,0


In [11]:
df_session[df_session[' id_session'] == 100130361][[' id_session', ' chat_start_time', ' chat_end_time', ' id_rep']]

,id_session,chat_start_time,chat_end_time,id_rep
79917,100130361,1494270076,1494271660,30000143
